In [ ]:
import tkinter as tk
from tkinter import ttk
from tkinter import filedialog as fd  # Dateibrowser-Modul importieren
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

def datei_waehlen_und_laden(system, entries, ax, canvas, status_label):
    """
    Öffnet ein Dateibrowser-Fenster, liest die Strukturdaten ein,
    befüllt das System-Dictionary und aktualisiert die GUI-Felder sowie den Plot.
    """
    # 1. Datei über Browser auswählen (Struktur aus deinem alten Code)
    filename = fd.askopenfilename(
        filetypes=[('Textdateien', '*.txt'), ('Alle Dateien', '*.*')]
    )

    if not filename:
        status_label.config(text="Status: Auswahl abgebrochen.")
        return

    try:
        with open(filename, 'r', encoding='utf-8') as file:
            lines = file.readlines()
    except Exception as e:
        status_label.config(text=f"Status: Fehler beim Lesen der Datei ({str(e)})")
        return

    # Kommentare entfernen und leere Zeilen filtern
    data = []
    for line in lines:
        clean_line = line.split('#')[0].strip()
        if clean_line:
            data.append(clean_line)

    try:
        # 2. Daten prozedural in das bestehende Dictionary parsen
        system['name'] = data[0]
        system['e_modul'] = float(data[1])
        system['streckgrenze'] = float(data[2])
        system['sicherheit'] = float(data[3])
        system['flaeche'] = float(data[4])
        
        current_idx = 5
        
        # Knoten einlesen
        num_nodes = int(data[current_idx])
        system['knoten'] = {}
        current_idx += 1
        for i in range(1, num_nodes + 1):
            coords = data[current_idx].split()
            system['knoten'][i] = [float(coords[0]), float(coords[1])]
            current_idx += 1
            
        # Elemente einlesen
        num_elements = int(data[current_idx])
        system['elemente'] = []
        current_idx += 1
        for _ in range(num_elements):
            nodes = data[current_idx].split()
            system['elemente'].append([int(nodes[0]), int(nodes[1])])
            current_idx += 1
            
        # Lasten einlesen
        num_loads = int(data[current_idx])
        system['lasten'] = []
        current_idx += 1
        for _ in range(num_loads):
            load_data = data[current_idx].split()
            system['lasten'].append([int(load_data[0]), float(load_data[1]), float(load_data[2])])
            current_idx += 1
            
        # Lager einlesen
        num_supports = int(data[current_idx])
        system['lager'] = []
        current_idx += 1
        for _ in range(num_supports):
            sup_data = data[current_idx].split()
            system['lager'].append([int(sup_data[0]), int(sup_data[1]), int(sup_data[2])])
            current_idx += 1

        # 3. GUI-Eingabefelder im Parameter-Tab aktualisieren (Struktur aus alten Code)
        entries['name'].delete(0, tk.END)
        entries['name'].insert(0, str(system['name']))
        
        entries['e_modul'].delete(0, tk.END)
        entries['e_modul'].insert(0, str(system['e_modul']))
        
        entries['streckgrenze'].delete(0, tk.END)
        entries['streckgrenze'].insert(0, str(system['streckgrenze']))
        
        entries['sicherheit'].delete(0, tk.END)
        entries['sicherheit'].insert(0, str(system['sicherheit']))
        
        entries['flaeche'].delete(0, tk.END)
        entries['flaeche'].insert(0, str(system['flaeche']))

        # 4. Grafik und Statuszeile aktualisieren
        aktualisiere_plot(system, ax, canvas)
        status_label.config(text=f"Status: Datei erfolgreich geladen ({filename})")

    except Exception as e:
        status_label.config(text=f"Status: Fehler beim Parsen der Datei ({str(e)})")


def aktualisiere_plot(system, ax, canvas):
    """Zeichnet die Knoten, Elemente, Lager und Lasten basierend auf den Systemdaten."""
    ax.clear()
    ax.set_title(system.get('name', 'Fachwerk (Keine Datei geladen)'))
    ax.set_xlabel('X [mm]')
    ax.set_ylabel('Y [mm]')
    ax.grid(True, linestyle='--', alpha=0.6)
    
    # Stäbe (Elemente) zeichnen
    for elem in system.get('elemente', []):
        n1 = system['knoten'][elem[0]]
        n2 = system['knoten'][elem[1]]
        ax.plot([n1[0], n2[0]], [n1[1], n2[1]], 'b-', zorder=1)

    # Knoten zeichnen
    for n_id, coords in system.get('knoten', {}).items():
        ax.plot(coords[0], coords[1], 'ro', zorder=2)
        if system.get('zeige_nummern', False):
        
        # A) Knoten-Nummern zeichnen
            for n_id, coords in system.get('knoten', {}).items():
                # offset points bedeutet: "Gehe vom Knoten exakt 5 Pixel nach rechts und oben"
                ax.annotate(str(n_id), xy=(coords[0], coords[1]), 
                            xytext=(5, 5), textcoords='offset points', 
                            color='darkred', fontsize=8, zorder=4)
                            
            # B) Stab-Nummern zeichnen
            # enumerate gibt uns 'i' (zählt von 0 hoch) und 'elem' (den Eintrag)
            for i, elem in enumerate(system.get('elemente', [])):
                stab_id = i + 1  # Die Stab-ID startet bei 1 (nicht bei 0)
                
                # Wir holen uns die X/Y-Koordinaten des Start- und Endknotens
                n1 = system['knoten'][elem[0]]
                n2 = system['knoten'][elem[1]]
                
                # Mittelpunkt des Stabes berechnen (Hälfte der Strecke)
                mid_x = (n1[0] + n2[0]) / 2.0
                mid_y = (n1[1] + n2[1]) / 2.0
                
                # Text exakt auf die Mitte setzen. Das 'S' hilft bei der Unterscheidung von den Knoten.
                ax.annotate(f"S{stab_id}", xy=(mid_x, mid_y), 
                            xytext=(0, 4), textcoords='offset points', 
                            color='darkblue', fontsize=8, fontweight='bold', zorder=4)

    # Lager zeichnen (Spitz-Dreiecke exakt anliegend am Knoten)
    for lager in system.get('lager', []):
        n_id, fix_x, fix_y = lager
        coords = system['knoten'][n_id]
        
        # X-Freiheitsgrad gesperrt -> Dreieck zeigt von links auf den Knoten
        if fix_x == 1:
            ax.annotate('', 
                        xy=(coords[0], coords[1]), 
                        xytext=(-15, 0), textcoords='offset points', # 15 Pixel Versatz nach links
                        arrowprops=dict(facecolor='green', edgecolor='green', 
                                        width=0.1, headwidth=14, headlength=14, shrink=0),
                        zorder=3)
            
        # Y-Freiheitsgrad gesperrt -> Dreieck zeigt von unten auf den Knoten
        if fix_y == 1:
            ax.annotate('', 
                        xy=(coords[0], coords[1]), 
                        xytext=(0, -15), textcoords='offset points', # 15 Pixel Versatz nach unten
                        arrowprops=dict(facecolor='green', edgecolor='green', 
                                        width=0.1, headwidth=14, headlength=14, shrink=0),
                        zorder=3)
                        
    # Lasten zeichnen
    for last in system.get('lasten', []):
        n_id, fx, fy = last
        coords = system['knoten'][n_id]
        ax.annotate(f'({fx}, {fy})', xy=(coords[0], coords[1]), 
                    xytext=(coords[0] + 5, coords[1] - 1), # Pfeil-Ursprung relativ zum Knoten anpassen
                    arrowprops=dict(facecolor='red', edgecolor='red', width=1, headwidth=5),
                    color='red', fontsize=8)

    # --- HIER IST DER NEUE CODE FÜR DIE SKALIERUNG ---
    
    # 1. Erzwinge das 1:1 Seitenverhältnis, ohne das Fenster zu verzerren
    ax.set_aspect('equal', adjustable='datalim') 
    
    # 2. Finde die maximalen und minimalen Koordinaten deiner Brücke
    if system.get('knoten'):
        x_vals = [c[0] for c in system['knoten'].values()]
        y_vals = [c[1] for c in system['knoten'].values()]
        
        min_x, max_x = min(x_vals), max(x_vals)
        min_y, max_y = min(y_vals), max(y_vals)
        
        # 3. Berechne die Spannweite (Breite und Höhe)
        span_x = max_x - min_x
        span_y = max_y - min_y
        
        # 4. Setze harte Achsengrenzen mit einem berechneten, dynamischen Puffer
        ax.set_xlim(min_x - (span_x * 0.1), max_x + (span_x * 0.1))
        # Y bekommt viel mehr Puffer (z.B. Faktor 1.5), damit die Pfeile oben hinpassen!
        ax.set_ylim(min_y - (span_y * 1.5), max_y + (span_y * 1.5)) 
    
    # --- ENDE NEUER CODE ---

    canvas.draw()
def parameter_uebernehmen(entries, system, ax, canvas, status_label):
    """Liest manuell geänderte Werte aus dem Parameter-Tab aus."""
    try:
        system['name'] = entries['name'].get()
        system['e_modul'] = float(entries['e_modul'].get())
        system['streckgrenze'] = float(entries['streckgrenze'].get())
        system['sicherheit'] = float(entries['sicherheit'].get())
        system['flaeche'] = float(entries['flaeche'].get())
        
        aktualisiere_plot(system, ax, canvas)
        status_label.config(text="Status: Parameter manuell aktualisiert.")
    except ValueError:
        status_label.config(text="Status: Fehler! Bitte gültige Zahlenwerte eingeben.")
def toggle_nummern(system, ax, canvas, status_label):
    """Prüft, ob eine Anzeige sinnvoll ist, und schaltet Knoten-/Stabnummern um."""
    anzahl_elemente = len(system.get('elemente', []))
    
    # 1. Sicherheits-Check: Gibt es überhaupt Daten?
    if anzahl_elemente == 0:
        status_label.config(text="Status: Fehler! Kein System geladen.")
        return
        
    # 2. Sinnhaftigkeits-Check: Sind es zu viele Elemente für den Bildschirm?
    # Ein Wert von 80 ist für dieses Fenster eine gute Grenze gegen "Zahlensalat"
    if anzahl_elemente > 80:
        status_label.config(text=f"Status: Fehler! Anzeige nicht sinnvoll (Zu viele Stäbe: {anzahl_elemente}).")
        return

    # 3. Zustand umschalten (True -> False -> True)
    system['zeige_nummern'] = not system.get('zeige_nummern', False)
    
    # Plot neu zeichnen und Status ausgeben
    aktualisiere_plot(system, ax, canvas)
    
    if system['zeige_nummern']:
        status_label.config(text="Status: Knoten- und Stabnummern eingeblendet.")
    else:
        status_label.config(text="Status: Nummern ausgeblendet zur besseren Übersicht.")

def aktualisiere_knoten_tab(tab_knoten, system, entries_knoten):
    """Baut die Eingabefelder für die Knoten dynamisch auf."""
    
    # 1. Zuerst alle alten Felder im Tab löschen (wichtig beim Laden einer neuen Datei!)
    for widget in tab_knoten.winfo_children():
        widget.destroy()
        
    entries_knoten.clear() # Altes Speicher-Dictionary leeren
    
    # Wenn noch keine Knoten da sind (Programmstart)
    if not system.get('knoten'):
        tk.Label(tab_knoten, text="Bitte zuerst eine Datei laden.").pack(pady=20)
        return

    # 2. Überschriften für die Tabelle erstellen
    tk.Label(tab_knoten, text="Knoten-ID", font=('Helvetica', 10, 'bold')).grid(row=0, column=0, padx=5, pady=5)
    tk.Label(tab_knoten, text="X [mm]", font=('Helvetica', 10, 'bold')).grid(row=0, column=1, padx=5, pady=5)
    tk.Label(tab_knoten, text="Y [mm]", font=('Helvetica', 10, 'bold')).grid(row=0, column=2, padx=5, pady=5)

    # 3. Für jeden Knoten automatisch eine neue Zeile generieren
    for i, (n_id, coords) in enumerate(system['knoten'].items()):
        row_idx = i + 1 # +1, weil Zeile 0 für die Überschriften reserviert ist
        
        # Label (z.B. "Knoten 1:")
        tk.Label(tab_knoten, text=f"Knoten {n_id}:").grid(row=row_idx, column=0, sticky="w", padx=5, pady=2)
        
        # Eingabefeld für X
        ent_x = tk.Entry(tab_knoten, width=10)
        ent_x.insert(0, str(coords[0]))
        ent_x.grid(row=row_idx, column=1, padx=5, pady=2)
        
        # Eingabefeld für Y
        ent_y = tk.Entry(tab_knoten, width=10)
        ent_y.insert(0, str(coords[1]))
        ent_y.grid(row=row_idx, column=2, padx=5, pady=2)
        
        # Wir speichern die Felder in unserem Dictionary ab, um später Werte auslesen zu können
        entries_knoten[n_id] = {'x': ent_x, 'y': ent_y}

def starte_gui():
    """Baut das Hauptfenster direkt auf, startet mit einem leeren System-Dictionary."""
    root = tk.Tk()
    root.title("FEM-Fachwerk Preprocessor")
    root.geometry("1000x650")

    # Initiales leeres System-Struktur-Dictionary definieren
    system_daten = {
        'name': 'Kein System geladen', 'e_modul': 0.0, 'streckgrenze': 0.0, 
        'sicherheit': 1.0, 'flaeche': 0.0, 'knoten': {}, 'elemente': [], 
        'lasten': [], 'lager': []
    }

    # --- Layout ---
    left_frame = tk.Frame(root, width=350, padx=10, pady=10)
    left_frame.pack(side=tk.LEFT, fill=tk.Y)
    left_frame.pack_propagate(False)

    right_frame = tk.Frame(root, padx=10, pady=10)
    right_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)

    status_label = tk.Label(root, text="Status: Bereit. Bitte Konfigurationsdatei laden.", anchor="w")
    status_label.pack(side=tk.BOTTOM, fill=tk.X)

    # --- HIER NEU: Datei öffnen Button ganz oben links platziert ---
    btn_load = tk.Button(left_frame, text="Datei öffnen...", bg="lightgray", font=('Helvetica', 10, 'bold'))
    btn_load.pack(fill=tk.X, pady=(0, 15))

    #---Button: Stabnummern anzeigen
    btn_toggle = tk.Button(left_frame, text="Nummern Ein/Aus", bg="white", 
                           command=lambda: toggle_nummern(system_daten, ax, canvas, status_label))
    btn_toggle.pack(fill=tk.X, pady=(0, 15))

    # --- Plot einrichten ---
    fig, ax = plt.subplots(figsize=(6, 5))
    canvas = FigureCanvasTkAgg(fig, master=right_frame)
    canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)

    # --- Tabs einrichten ---
    notebook = ttk.Notebook(left_frame)
    notebook.pack(fill=tk.BOTH, expand=True)

    tab_param = ttk.Frame(notebook)
    tab_knoten = ttk.Frame(notebook)
    tab_elemente = ttk.Frame(notebook)
    tab_lasten = ttk.Frame(notebook)
    tab_lager = ttk.Frame(notebook)

    notebook.add(tab_param, text="Parameter")
    notebook.add(tab_knoten, text="Knoten")
    notebook.add(tab_elemente, text="Elemente")
    notebook.add(tab_lasten, text="Lasten")
    notebook.add(tab_lager, text="Lager")

    entries_knoten = {} 
    aktualisiere_knoten_tab(tab_knoten, system_daten, entries_knoten)

    # --- Tab: Parameter befüllen ---
    entries = {}
    labels = ["Titel:", "E-Modul [N/mm²]:", "Streckgrenze [N/mm²]:", "Sicherheit:", "Querschnitt A [mm²]:"]
    keys = ['name', 'e_modul', 'streckgrenze', 'sicherheit', 'flaeche']

    for i, (label_text, key) in enumerate(zip(labels, keys)):
        tk.Label(tab_param, text=label_text).grid(row=i, column=0, sticky="w", pady=10, padx=5)
        ent = tk.Entry(tab_param, width=20)
        ent.insert(0, str(system_daten[key]))
        ent.grid(row=i, column=1, pady=10, padx=5)
        entries[key] = ent

    # Button für manuelle Parameteränderungen
    tk.Button(tab_param, text="Parameter übernehmen", 
              command=lambda: parameter_uebernehmen(entries, system_daten, ax, canvas, status_label)
             ).grid(row=len(labels), column=0, columnspan=2, pady=20)

    # Jetzt den "Datei öffnen..." Button oben mit der neuen Funktion verknüpfen
    btn_load.config(command=lambda: datei_waehlen_und_laden(system_daten, entries, ax, canvas, status_label))
    # Initial leeren Plot zeichnen
    aktualisiere_plot(system_daten, ax, canvas)
    
    root.mainloop()

# --- Programm startet direkt und öffnet die GUI ---
starte_gui()

In [2]:
import sys
print(sys.executable)

c:\Users\leoja\AppData\Local\Programs\Python\Python314\python.exe
